# Asset Embeddings: Three-Quarter Pipeline Validation

This notebook is the clean integration test for the WRDS asset-embeddings replication. It runs the same checkpointed pipeline for 2005Q1, 2014Q4, and 2024Q4 before the workflow is expanded to all 80 quarters from 2005Q1 through 2024Q4.

The validation covers raw S12 and S34 holdings, historical security linkage, historical fund–manager overlap, Jensen-style firm characteristics, paper-consistent sample filters, sparse matrices, RS-Binary and RS-L-Min embeddings, and the relative-valuation benchmark from *Asset Embeddings* by Gabaix, Koijen, Richmond, and Yogo (2025).

All WRDS extracts and derived artifacts are stored locally. Existing valid files are reused, and expensive stages are checkpointed by quarter. Credentials are requested interactively and are never written to disk.


## 1. Imports, project paths, and validation configuration

The three dates deliberately span the beginning, middle, and end of the sample. Set `FORCE_REBUILD` to `True` only when a derived validation artifact must be regenerated. Immutable raw WRDS extracts are never overwritten automatically.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "Asset_Embeddings":
    candidate = PROJECT_ROOT / "Projects" / "Asset_Embeddings"
    if candidate.exists():
        PROJECT_ROOT = candidate.resolve()
if PROJECT_ROOT.name != "Asset_Embeddings":
    raise RuntimeError("Open this notebook from the Asset_Embeddings project directory.")

os.chdir(PROJECT_ROOT)

LOCAL_PACKAGES = PROJECT_ROOT / ".packages"
if str(LOCAL_PACKAGES) not in sys.path:
    sys.path.insert(0, str(LOCAL_PACKAGES))

from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


VALIDATION_QUARTERS = [(2005, 1), (2014, 4), (2024, 4)]
HOLDINGS_PARTITIONS = 16
MINIMUM_DEGREE = 20
MAXIMUM_POSITION_WEIGHT = 0.75
MAXIMUM_EMBEDDING_DIMENSION = 128
RANDOM_SEED = 202451
TEST_FRACTION = 0.20
RIDGE_ALPHAS = np.logspace(-6, 6, 49)
FORCE_REBUILD = False
PIPELINE_VERSION = 2

VALIDATION_ROOT = PROJECT_ROOT / "validation_3q"
for folder in ["manifests", "processed", "matrices", "embeddings", "benchmarks", "figures", "logs"]:
    (VALIDATION_ROOT / folder).mkdir(parents=True, exist_ok=True)


def quarter_end(year, quarter):
    return pd.Period(f"{year}Q{quarter}", freq="Q").end_time.normalize()


def quarter_label(year, quarter):
    return f"{year}q{quarter}"


configuration = {
    "quarters": VALIDATION_QUARTERS,
    "holdings_partitions": HOLDINGS_PARTITIONS,
    "minimum_degree": MINIMUM_DEGREE,
    "maximum_position_weight": MAXIMUM_POSITION_WEIGHT,
    "maximum_embedding_dimension": MAXIMUM_EMBEDDING_DIMENSION,
    "random_seed": RANDOM_SEED,
    "test_fraction": TEST_FRACTION,
    "pipeline_version": PIPELINE_VERSION,
}
with open(VALIDATION_ROOT / "manifests" / "configuration.json", "w") as handle:
    json.dump(configuration, handle, indent=2)

print(f"Project root: {PROJECT_ROOT}")
print(f"Validation quarters: {VALIDATION_QUARTERS}")
print(f"Validation output root: {VALIDATION_ROOT}")


## 2. Open or restore the WRDS connection

This cell reuses an active connection when possible. Otherwise, it requests the WRDS username and lets the WRDS client securely prompt for authentication.


In [ ]:
import wrds


def wrds_connection_is_active(connection):
    try:
        connection.raw_sql("select 1 as connection_test")
        return True
    except Exception:
        return False


if "db" in globals() and wrds_connection_is_active(db):
    print("Existing WRDS connection is active.")
else:
    WRDS_USERNAME = os.getenv("WRDS_USERNAME") or input("WRDS username: ").strip()
    if not WRDS_USERNAME:
        raise ValueError("A WRDS username is required.")
    db = wrds.Connection(wrds_username=WRDS_USERNAME)
    print("WRDS connection established.")


## 3. Checkpoint and data-normalization utilities

These helpers standardize identifiers, save Parquet files, validate checkpoints, and avoid repeating completed work.


In [ ]:
def normalize_cusip(series):
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[^0-9A-Z]", "", regex=True)
        .str[:8]
    )


def parquet_is_valid(path, required_columns=None):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        frame = pd.read_parquet(path)
    except Exception:
        return False
    if required_columns and not set(required_columns).issubset(frame.columns):
        return False
    return True


def save_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(path, index=False)


def market_equity_weighted_average(group, variable, weight="me"):
    valid = group[variable].notna() & group[weight].notna() & group[weight].gt(0)
    if not valid.any():
        return np.nan
    return float(np.average(group.loc[valid, variable], weights=group.loc[valid, weight]))


def recompute_portfolio_fields(frame):
    frame = frame.copy()
    frame["portfolio_value"] = frame.groupby("investor_key")["dollar_holding"].transform("sum")
    frame["portfolio_weight"] = frame["dollar_holding"] / frame["portfolio_value"]
    frame["log_dollar_holding"] = np.log(frame["dollar_holding"])
    frame["firm_rank"] = frame.groupby("investor_key")["dollar_holding"].rank(method="first", ascending=False).astype("int32")
    return frame


def iterative_degree_filter(frame, minimum_degree=20):
    current = frame.copy()
    audit = []
    iteration = 0
    while True:
        iteration += 1
        rows_before = len(current)
        investor_degree = current.groupby("investor_key")["permco"].nunique()
        firm_degree = current.groupby("permco")["investor_key"].nunique()
        current = current[
            current["investor_key"].isin(investor_degree[investor_degree >= minimum_degree].index)
            & current["permco"].isin(firm_degree[firm_degree >= minimum_degree].index)
        ].copy()
        audit.append({
            "iteration": iteration,
            "rows_before": rows_before,
            "rows_after": len(current),
            "investors_after": current["investor_key"].nunique(),
            "firms_after": current["permco"].nunique(),
        })
        if len(current) == rows_before:
            break
        if current.empty:
            raise ValueError("Degree filtering removed the entire sample.")
    return current, pd.DataFrame(audit)


## 4. Historical point-in-time CRSP security crosswalk

For each validation quarter, the active CRSP name interval maps the reported eight-character CUSIP to PERMNO and PERMCO. This prevents the 2024 crosswalk from being applied retrospectively.


In [ ]:
def obtain_crsp_crosswalk(year, quarter):
    date = quarter_end(year, quarter)
    label = quarter_label(year, quarter)
    path = PROJECT_ROOT / "data" / "crosswalks" / "crsp" / f"year={year}" / f"quarter={quarter}" / f"stocknames_{label}.parquet"
    required = ["ncusip", "permno", "permco", "shrcd"]
    if parquet_is_valid(path, required) and not FORCE_REBUILD:
        frame = pd.read_parquet(path)
    else:
        query = f"""
            select ncusip, permno, permco, namedt, nameenddt,
                   shrcd, exchcd, comnam, ticker
            from crsp_a_stock.stocknames
            where namedt <= '{date:%Y-%m-%d}'
              and (nameenddt >= '{date:%Y-%m-%d}' or nameenddt is null)
        """
        frame = db.raw_sql(query, date_cols=["namedt", "nameenddt"])
        frame["ncusip"] = normalize_cusip(frame["ncusip"])
        for column in ["permno", "permco", "shrcd", "exchcd"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=["ncusip", "permno", "permco"]).copy()
        counts = frame.groupby("ncusip")["permno"].nunique()
        ambiguous = set(counts[counts > 1].index)
        if ambiguous:
            frame = frame[~frame["ncusip"].isin(ambiguous)].copy()
        save_parquet(frame, path)
    return frame


crosswalk_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    crosswalk = obtain_crsp_crosswalk(year, quarter)
    crosswalk_inventory.append({
        "year": year, "quarter": quarter, "rows": len(crosswalk),
        "permnos": crosswalk["permno"].nunique(), "permcos": crosswalk["permco"].nunique(),
        "ordinary_common_rows": int(crosswalk["shrcd"].isin([10, 11]).sum()),
    })
display(pd.DataFrame(crosswalk_inventory))


## 5. Partitioned S12 and S34 downloads

Holdings are pulled from the dedicated `tr_mutualfunds.s12` and `tr_13f.s34` tables. The baseline uses strict quarter-end observations with `fdate = rdate = quarter end`. Sixteen investor-ID partitions provide restartable downloads and bounded memory use.


In [ ]:
def holdings_specification(source):
    if source == "s12":
        return "tr_mutualfunds.s12", "fundno"
    if source == "s34":
        return "tr_13f.s34", "mgrno"
    raise ValueError(source)


def obtain_raw_holdings(source, year, quarter):
    table, investor_column = holdings_specification(source)
    date = quarter_end(year, quarter)
    directory = PROJECT_ROOT / "data" / "raw" / source / f"year={year}" / f"quarter={quarter}"
    directory.mkdir(parents=True, exist_ok=True)
    audit_rows = []
    for partition in range(HOLDINGS_PARTITIONS):
        path = directory / f"part-{partition:02d}-of-{HOLDINGS_PARTITIONS:02d}.parquet"
        start = time.perf_counter()
        if parquet_is_valid(path, ["fdate", "rdate", investor_column, "cusip", "shares", "prc"]):
            frame = pd.read_parquet(path)
            status = "checkpoint"
        else:
            query = f"""
                select *
                from {table}
                where fdate = '{date:%Y-%m-%d}'
                  and rdate = '{date:%Y-%m-%d}'
                  and mod(cast({investor_column} as bigint), {HOLDINGS_PARTITIONS}) = {partition}
            """
            frame = db.raw_sql(query, date_cols=["fdate", "rdate", "prdate"])
            save_parquet(frame, path)
            status = "downloaded"
        audit_rows.append({
            "source": source, "year": year, "quarter": quarter, "partition": partition,
            "status": status, "rows": len(frame),
            "investors": frame[investor_column].nunique(),
            "securities": frame["cusip"].nunique(),
            "seconds": time.perf_counter() - start,
            "file_mb": path.stat().st_size / (1024 ** 2),
        })
    return directory, pd.DataFrame(audit_rows)


download_audits = []
for year, quarter in VALIDATION_QUARTERS:
    for source in ["s12", "s34"]:
        _, audit = obtain_raw_holdings(source, year, quarter)
        download_audits.append(audit)
download_audit = pd.concat(download_audits, ignore_index=True)
save_parquet(download_audit, VALIDATION_ROOT / "manifests" / "holdings_download_audit.parquet")
display(download_audit.groupby(["year", "quarter", "source"], as_index=False).agg(rows=("rows", "sum"), file_mb=("file_mb", "sum"), downloaded=("status", lambda x: int((x == "downloaded").sum()))))


## 6. Historical S12 fund-to-S34 manager mappings

The `s12type5` crosswalk is downloaded separately for each quarter. An S34 parent manager is removed from the combined sample only when it is linked to an active S12 fund in that same quarter.


In [ ]:
def obtain_s12_s34_mapping(year, quarter):
    date = quarter_end(year, quarter)
    label = quarter_label(year, quarter)
    directory = PROJECT_ROOT / "data" / "crosswalks" / "s12_s34" / f"year={year}" / f"quarter={quarter}"
    path = directory / f"s12type5_{label}.parquet"
    if parquet_is_valid(path, ["fdate", "fundno", "mgrcocd"]):
        return pd.read_parquet(path)
    legacy = PROJECT_ROOT / "data" / "crosswalks" / "s12_s34" / f"s12type5_{label}_raw.parquet"
    if parquet_is_valid(legacy, ["fdate", "fundno", "mgrcocd"]):
        frame = pd.read_parquet(legacy)
    else:
        query = f"""
            select fdate, fundno, fundname, mgrco, mgrcocd
            from tr_mutualfunds.s12type5
            where fdate = '{date:%Y-%m-%d}'
        """
        frame = db.raw_sql(query, date_cols=["fdate"])
    save_parquet(frame, path)
    return frame


mapping_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    mapping = obtain_s12_s34_mapping(year, quarter)
    mapping_inventory.append({"year": year, "quarter": quarter, "rows": len(mapping), "funds": mapping["fundno"].nunique(), "managers": mapping["mgrcocd"].nunique()})
display(pd.DataFrame(mapping_inventory))


## 7. Jensen-style firm characteristics and exact size categories

The contributed `global_factor` table supplies the Appendix D.1.2 variables: `beta_60m`, `at_gr1`, `gp_at`, and `div_at`, plus book and market equity. Security observations are aggregated to PERMCO using market-equity weights. Firms must have at least one small, large, or mega security; nano and micro firms are excluded.


In [ ]:
def obtain_jkp_security_data(year, quarter):
    date = quarter_end(year, quarter)
    label = quarter_label(year, quarter)
    directory = PROJECT_ROOT / "data" / "raw" / "jkp_characteristics" / f"year={year}" / f"quarter={quarter}"
    path = directory / f"global_factor_paper_exact_{label}.parquet"
    required = ["permno", "permco", "me", "book_equity", "beta_60m", "at_gr1", "gp_at", "div_at"]
    if parquet_is_valid(path, required):
        return pd.read_parquet(path)
    query = f"""
        select id, permno, permco, gvkey, iid, excntry, common, primary_sec,
               obs_main, exch_main, crsp_shrcd, crsp_exchcd, date, eom,
               size_grp, me, me_company, book_equity,
               beta_60m, at_gr1, gp_at, div_at
        from contrib_global_factor.global_factor
        where eom = '{date:%Y-%m-%d}' and excntry = 'USA'
    """
    frame = db.raw_sql(query, date_cols=["date", "eom"])
    save_parquet(frame, path)
    return frame


def construct_jkp_firm_panel(year, quarter):
    label = quarter_label(year, quarter)
    output = VALIDATION_ROOT / "processed" / label / "jkp_firm_panel.parquet"
    if parquet_is_valid(output, ["permco", "beta_60m_paper", "at_gr1_paper", "gp_at_paper", "div_at_paper"]) and not FORCE_REBUILD:
        return pd.read_parquet(output)
    frame = obtain_jkp_security_data(year, quarter).copy()
    numeric = ["permno", "permco", "common", "primary_sec", "obs_main", "exch_main", "crsp_shrcd", "me", "me_company", "book_equity", "beta_60m", "at_gr1", "gp_at", "div_at"]
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(subset=["permno", "permco"]).copy()
    frame["permno"] = frame["permno"].astype("int64")
    frame["permco"] = frame["permco"].astype("int64")
    frame["size_group_clean"] = frame["size_grp"].astype("string").str.lower().str.strip()
    frame = frame[frame["common"].eq(1) | frame["crsp_shrcd"].isin([10, 11])].copy()
    frame["eligible_size"] = frame["size_group_clean"].isin(["small", "large", "mega"])
    rows = []
    for permco, group in frame.groupby("permco", sort=True):
        positive_me = group.loc[group["me"].gt(0), "me"]
        positive_be = group.loc[group["book_equity"].gt(0), "book_equity"]
        row = {
            "permco": int(permco),
            "eligible_size_security": bool(group["eligible_size"].any()),
            "underlying_permnos": int(group["permno"].nunique()),
            "jkp_firm_market_equity": float(positive_me.sum()) if len(positive_me) else np.nan,
            "jkp_book_equity": float(positive_be.max()) if len(positive_be) else np.nan,
        }
        for variable in ["beta_60m", "at_gr1", "gp_at", "div_at"]:
            row[variable] = market_equity_weighted_average(group, variable)
        rows.append(row)
    firms = pd.DataFrame(rows)
    firms = firms[firms["eligible_size_security"] & firms["jkp_book_equity"].gt(0)].copy()
    firms = firms.dropna(subset=["beta_60m", "at_gr1", "gp_at", "div_at"]).copy()
    firms["beta_60m_paper"] = firms["beta_60m"]
    for variable in ["at_gr1", "gp_at"]:
        lower, upper = firms[variable].quantile([0.01, 0.99])
        firms[f"{variable}_paper"] = firms[variable].clip(lower=lower, upper=upper)
    firms["div_at_paper"] = firms["div_at"].clip(upper=firms["div_at"].quantile(0.99))
    save_parquet(firms, output)
    return firms


jkp_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    firms = construct_jkp_firm_panel(year, quarter)
    jkp_inventory.append({"year": year, "quarter": quarter, "complete_case_firms": len(firms), "eligible_permcos": firms["permco"].nunique()})
display(pd.DataFrame(jkp_inventory))


## 8. Link holdings, aggregate to firms, and construct the combined sample

CUSIPs are matched using the quarter-specific CRSP crosswalk. S12 shares are actual share counts, so dollar holdings equal shares times absolute price. Securities are aggregated to PERMCO. The combined sample removes overlapping S34 parents, excludes portfolios with a largest position above 75%, and iteratively requires at least 20 firms per investor and 20 investors per firm.


In [ ]:
def link_and_aggregate_source(source, year, quarter):
    label = quarter_label(year, quarter)
    output = VALIDATION_ROOT / "processed" / label / f"{source}_firm_holdings_prefilter.parquet"
    if parquet_is_valid(output, ["investor_key", "permco", "dollar_holding"]) and not FORCE_REBUILD:
        return pd.read_parquet(output)
    raw_directory, _ = obtain_raw_holdings(source, year, quarter)
    crosswalk = obtain_crsp_crosswalk(year, quarter).copy()
    crosswalk["ncusip"] = normalize_cusip(crosswalk["ncusip"])
    crosswalk = crosswalk[crosswalk["shrcd"].isin([10, 11])].copy()
    crosswalk = crosswalk.drop_duplicates("ncusip")
    investor_column = "fundno" if source == "s12" else "mgrno"
    name_column = "fundname" if source == "s12" else "mgrname"
    pieces = []
    for path in sorted(raw_directory.glob("part-*-of-*.parquet")):
        raw = pd.read_parquet(path)
        raw["cusip8"] = normalize_cusip(raw["cusip"])
        raw["shares"] = pd.to_numeric(raw["shares"], errors="coerce")
        raw["prc"] = pd.to_numeric(raw["prc"], errors="coerce")
        raw[investor_column] = pd.to_numeric(raw[investor_column], errors="coerce")
        linked = raw.merge(crosswalk[["ncusip", "permno", "permco", "comnam"]], left_on="cusip8", right_on="ncusip", how="inner", validate="many_to_one")
        linked["dollar_holding"] = linked["shares"] * linked["prc"].abs()
        linked = linked[linked["dollar_holding"].gt(0) & linked[investor_column].notna()].copy()
        linked["investor_id"] = linked[investor_column].astype("int64")
        linked["investor_key"] = source + ":" + linked["investor_id"].astype(str)
        linked["investor_name"] = linked[name_column].astype("string")
        linked["permco"] = linked["permco"].astype("int64")
        pieces.append(linked[["fdate", "rdate", "investor_id", "investor_key", "investor_name", "permco", "comnam", "permno", "dollar_holding"]])
    security = pd.concat(pieces, ignore_index=True)
    firm = security.groupby(["fdate", "rdate", "investor_id", "investor_key", "investor_name", "permco"], as_index=False, dropna=False).agg(
        firm_name=("comnam", "first"), dollar_holding=("dollar_holding", "sum"),
        underlying_security_rows=("permno", "size"), underlying_permnos=("permno", "nunique"),
    )
    firm.insert(0, "source", source)
    save_parquet(firm, output)
    return firm


def construct_final_combined_holdings(year, quarter):
    label = quarter_label(year, quarter)
    output = VALIDATION_ROOT / "processed" / label / "combined_final_holdings.parquet"
    audit_path = VALIDATION_ROOT / "processed" / label / "filter_audit.parquet"
    if parquet_is_valid(output, ["investor_key", "permco", "portfolio_weight", "pipeline_version"]) and not FORCE_REBUILD:
        return pd.read_parquet(output), pd.read_parquet(audit_path)
    s12 = link_and_aggregate_source("s12", year, quarter)
    s34 = link_and_aggregate_source("s34", year, quarter)
    mapping = obtain_s12_s34_mapping(year, quarter).copy()
    mapping["fundno"] = pd.to_numeric(mapping["fundno"], errors="coerce")
    mapping["mgrcocd"] = pd.to_numeric(mapping["mgrcocd"], errors="coerce")
    raw_s12_directory, _ = obtain_raw_holdings("s12", year, quarter)
    active_s12 = set()
    for raw_path in sorted(raw_s12_directory.glob("part-*-of-*.parquet")):
        raw_fund_ids = pd.read_parquet(raw_path, columns=["fundno"])["fundno"]
        active_s12.update(pd.to_numeric(raw_fund_ids, errors="coerce").dropna().astype("int64"))
    active_s34 = set(s34["investor_id"])
    overlapping_managers = set(mapping.loc[mapping["fundno"].isin(active_s12), "mgrcocd"].dropna().astype("int64")) & active_s34
    s34 = s34[~s34["investor_id"].isin(overlapping_managers)].copy()
    combined = pd.concat([s12, s34], ignore_index=True)
    combined["pipeline_version"] = PIPELINE_VERSION
    eligible_firms = construct_jkp_firm_panel(year, quarter)[["permco", "jkp_firm_market_equity"]]
    combined = combined.merge(eligible_firms, on="permco", how="inner", validate="many_to_one")
    combined = recompute_portfolio_fields(combined)
    concentration = combined.groupby("investor_key")["portfolio_weight"].max()
    concentrated = set(concentration[concentration > MAXIMUM_POSITION_WEIGHT].index)
    combined = combined[~combined["investor_key"].isin(concentrated)].copy()
    combined, iteration_audit = iterative_degree_filter(combined, MINIMUM_DEGREE)
    combined = recompute_portfolio_fields(combined)
    audit = pd.DataFrame({
        "year": [year], "quarter": [quarter], "active_s34_managers_removed": [len(overlapping_managers)],
        "concentrated_investors_removed": [len(concentrated)], "final_rows": [len(combined)],
        "final_investors": [combined["investor_key"].nunique()], "final_firms": [combined["permco"].nunique()],
        "degree_iterations": [len(iteration_audit)], "pipeline_version": [PIPELINE_VERSION],
        "s34_overlap_rule": ["mapped_to_any_raw_active_s12_fund"],
    })
    save_parquet(combined, output)
    save_parquet(audit, audit_path)
    save_parquet(iteration_audit, VALIDATION_ROOT / "processed" / label / "degree_iteration_audit.parquet")
    return combined, audit


filter_audits = []
for year, quarter in VALIDATION_QUARTERS:
    holdings, audit = construct_final_combined_holdings(year, quarter)
    filter_audits.append(audit)
filter_audit = pd.concat(filter_audits, ignore_index=True)
save_parquet(filter_audit, VALIDATION_ROOT / "manifests" / "quarterly_filter_audit.parquet")
display(filter_audit)


## 9. Sparse holdings matrices

The final combined holdings become aligned investor-by-firm matrices for binary ownership and positive log dollar holdings. Mapping tables make every row and column recoverable.


In [ ]:
def construct_sparse_matrices(year, quarter):
    label = quarter_label(year, quarter)
    directory = VALIDATION_ROOT / "matrices" / label
    binary_path = directory / "binary.npz"
    log_path = directory / "log_dollar_holdings_positive.npz"
    investor_path = directory / "investor_mapping.parquet"
    firm_path = directory / "firm_mapping.parquet"
    summary_path = directory / "matrix_summary.parquet"
    if binary_path.exists() and log_path.exists() and parquet_is_valid(investor_path) and parquet_is_valid(firm_path) and parquet_is_valid(summary_path, ["pipeline_version"]) and not FORCE_REBUILD:
        return sparse.load_npz(binary_path), sparse.load_npz(log_path), pd.read_parquet(investor_path), pd.read_parquet(firm_path)
    holdings, _ = construct_final_combined_holdings(year, quarter)
    directory.mkdir(parents=True, exist_ok=True)
    investor_mapping = holdings[["investor_key", "source", "investor_id", "investor_name"]].drop_duplicates("investor_key").sort_values("investor_key").reset_index(drop=True)
    investor_mapping["investor_index"] = np.arange(len(investor_mapping), dtype="int32")
    firm_mapping = holdings[["permco", "firm_name"]].drop_duplicates("permco").sort_values("permco").reset_index(drop=True)
    firm_mapping["firm_index"] = np.arange(len(firm_mapping), dtype="int32")
    indexed = holdings.merge(investor_mapping[["investor_key", "investor_index"]], on="investor_key", validate="many_to_one").merge(firm_mapping[["permco", "firm_index"]], on="permco", validate="many_to_one")
    rows = indexed["investor_index"].to_numpy()
    cols = indexed["firm_index"].to_numpy()
    shape = (len(investor_mapping), len(firm_mapping))
    binary = sparse.csr_matrix((np.ones(len(indexed), dtype="float32"), (rows, cols)), shape=shape)
    log_values = np.log(indexed["dollar_holding"].to_numpy(dtype="float64")).astype("float32")
    log_matrix = sparse.csr_matrix((log_values, (rows, cols)), shape=shape)
    sparse.save_npz(binary_path, binary, compressed=True)
    sparse.save_npz(log_path, log_matrix, compressed=True)
    save_parquet(investor_mapping, investor_path)
    save_parquet(firm_mapping, firm_path)
    summary = pd.DataFrame({"investors": [shape[0]], "firms": [shape[1]], "nonzero_holdings": [binary.nnz], "density": [binary.nnz / (shape[0] * shape[1])], "pipeline_version": [PIPELINE_VERSION]})
    save_parquet(summary, summary_path)
    return binary, log_matrix, investor_mapping, firm_mapping


matrix_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    binary, _, investors, firms = construct_sparse_matrices(year, quarter)
    matrix_inventory.append({"year": year, "quarter": quarter, "investors": len(investors), "firms": len(firms), "nonzero": binary.nnz, "density": binary.nnz / np.prod(binary.shape)})
display(pd.DataFrame(matrix_inventory))


## 10. RS-Binary and RS-L-Min embeddings

RS-Binary applies centered PCA to the binary holdings matrix. RS-L-Min removes investor and firm fixed effects from positive log holdings, replaces each missing position with that investor's minimum observed residual, and applies centered PCA. Asset embeddings are normalized so their cross-sectional Gram matrix is approximately the identity.


In [ ]:
def normalized_asset_embeddings(pca, number_of_firms):
    return (pca.components_.T * np.sqrt(number_of_firms)).astype("float32")


def fit_embeddings(year, quarter):
    label = quarter_label(year, quarter)
    binary, log_sparse, investor_mapping, firm_mapping = construct_sparse_matrices(year, quarter)
    number_of_components = min(MAXIMUM_EMBEDDING_DIMENSION, binary.shape[0] - 1, binary.shape[1] - 1)
    outputs = {}
    for model in ["rs_binary", "rs_l_min"]:
        directory = VALIDATION_ROOT / "embeddings" / label / model
        path = directory / "asset_embeddings_normalized_labeled.parquet"
        fit_summary_path = directory / "fit_summary.parquet"
        if parquet_is_valid(path, ["permco", "embedding_001"]) and parquet_is_valid(fit_summary_path, ["pipeline_version"]) and not FORCE_REBUILD:
            outputs[model] = pd.read_parquet(path)
            continue
        directory.mkdir(parents=True, exist_ok=True)
        if model == "rs_binary":
            dense = binary.toarray().astype("float32")
        else:
            coo = log_sparse.tocoo()
            observed_counts_i = np.bincount(coo.row, minlength=log_sparse.shape[0])
            investor_sums = np.bincount(coo.row, weights=coo.data, minlength=log_sparse.shape[0])
            investor_fe = investor_sums / observed_counts_i
            investor_demeaned = coo.data - investor_fe[coo.row]
            observed_counts_a = np.bincount(coo.col, minlength=log_sparse.shape[1])
            firm_sums = np.bincount(coo.col, weights=investor_demeaned, minlength=log_sparse.shape[1])
            firm_fe = firm_sums / observed_counts_a
            residual = investor_demeaned - firm_fe[coo.col]
            investor_minimum = np.full(log_sparse.shape[0], np.inf, dtype="float64")
            np.minimum.at(investor_minimum, coo.row, residual)
            dense = np.repeat(investor_minimum[:, None], log_sparse.shape[1], axis=1).astype("float32")
            dense[coo.row, coo.col] = residual.astype("float32")
        start = time.perf_counter()
        pca = PCA(n_components=number_of_components, svd_solver="randomized", random_state=RANDOM_SEED)
        pca.fit(dense)
        asset = normalized_asset_embeddings(pca, dense.shape[1])
        labeled = firm_mapping[["permco", "firm_name", "firm_index"]].copy()
        for component in range(number_of_components):
            labeled[f"embedding_{component + 1:03d}"] = asset[:, component]
        save_parquet(labeled, path)
        fit_summary = pd.DataFrame({
            "model": [model], "investors": [dense.shape[0]], "firms": [dense.shape[1]],
            "components": [number_of_components], "fit_seconds": [time.perf_counter() - start],
            "cumulative_explained_variance_128": [float(pca.explained_variance_ratio_.sum())],
            "pipeline_version": [PIPELINE_VERSION],
        })
        save_parquet(fit_summary, fit_summary_path)
        np.save(directory / "explained_variance_ratio.npy", pca.explained_variance_ratio_)
        outputs[model] = labeled
        del dense
    return outputs


embedding_inventory = []
for year, quarter in VALIDATION_QUARTERS:
    outputs = fit_embeddings(year, quarter)
    for model, frame in outputs.items():
        embedding_inventory.append({"year": year, "quarter": quarter, "model": model, "firms": len(frame), "dimensions": len([c for c in frame if c.startswith("embedding_")])})
display(pd.DataFrame(embedding_inventory))


## 11. Paper-style relative-valuation benchmark by quarter

Within the final common sample, log market equity is regressed on log book equity and the residual becomes relative valuation. A fixed 80/20 firm split is shared by Gaussian placebos, Beta, four characteristics, RS-Binary, RS-L-Min, and embedding-plus-characteristic models. Ridge penalties are chosen by ten-fold cross-validation. The paper's variance-based test R-squared is calculated separately in every quarter.


In [ ]:
def run_valuation_benchmark(year, quarter):
    label = quarter_label(year, quarter)
    directory = VALIDATION_ROOT / "benchmarks" / label
    result_path = directory / "relative_valuation_results.parquet"
    if parquet_is_valid(result_path, ["model", "test_variance_r2", "pipeline_version"]) and not FORCE_REBUILD:
        return pd.read_parquet(result_path)
    directory.mkdir(parents=True, exist_ok=True)
    characteristics = construct_jkp_firm_panel(year, quarter).copy()
    embeddings = fit_embeddings(year, quarter)
    panel = characteristics.copy()
    for model, frame in embeddings.items():
        columns = sorted(c for c in frame if c.startswith("embedding_"))
        renamed = frame[["permco"] + columns].rename(columns={c: f"{model}_{c}" for c in columns})
        panel = panel.merge(renamed, on="permco", how="inner", validate="one_to_one")
    panel = panel[panel["jkp_firm_market_equity"].gt(0) & panel["jkp_book_equity"].gt(0)].sort_values("permco").reset_index(drop=True)
    panel["log_me"] = np.log(panel["jkp_firm_market_equity"])
    panel["log_be"] = np.log(panel["jkp_book_equity"])
    valuation = LinearRegression().fit(panel[["log_be"]], panel["log_me"])
    panel["relative_valuation"] = panel["log_me"] - valuation.predict(panel[["log_be"]])
    n = len(panel)
    rng = np.random.default_rng(RANDOM_SEED + year * 10 + quarter)
    order = rng.permutation(n)
    n_test = int(np.ceil(TEST_FRACTION * n))
    test_positions, train_positions = order[:n_test], order[n_test:]
    placebo = rng.standard_normal((n, 10))
    for index in range(10):
        panel[f"placebo_{index + 1:03d}"] = placebo[:, index]
    characteristics_columns = ["beta_60m_paper", "at_gr1_paper", "gp_at_paper", "div_at_paper"]
    binary = [f"rs_binary_embedding_{i:03d}" for i in range(1, 11)]
    lmin = [f"rs_l_min_embedding_{i:03d}" for i in range(1, 11)]
    placebo_columns = [f"placebo_{i:03d}" for i in range(1, 11)]
    models = {
        "placebo_4": placebo_columns[:4], "placebo_10": placebo_columns,
        "beta": ["beta_60m_paper"], "characteristics_4": characteristics_columns,
        "rs_binary_4": binary[:4], "rs_binary_10": binary,
        "rs_l_min_4": lmin[:4], "rs_l_min_10": lmin,
        "rs_binary_10_plus_characteristics": binary + characteristics_columns,
        "rs_l_min_10_plus_characteristics": lmin + characteristics_columns,
    }
    y = panel["relative_valuation"].to_numpy(dtype="float64")
    y_train, y_test = y[train_positions], y[test_positions]
    cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
    rows = []
    predictions = []
    for model, columns in models.items():
        X = panel[columns].to_numpy(dtype="float64")
        scaler = StandardScaler().fit(X[train_positions])
        X_train, X_test = scaler.transform(X[train_positions]), scaler.transform(X[test_positions])
        ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=cv, scoring="neg_mean_squared_error").fit(X_train, y_train)
        prediction = ridge.predict(X_test)
        residual = y_test - prediction
        rows.append({
            "year": year, "quarter": quarter, "model": model, "predictors": len(columns),
            "firms": n, "training_firms": len(train_positions), "test_firms": len(test_positions),
            "selected_alpha": float(ridge.alpha_),
            "test_variance_r2": float(1 - np.var(residual) / np.var(y_test)),
            "test_conventional_r2": float(r2_score(y_test, prediction)),
            "test_prediction_correlation": float(np.corrcoef(y_test, prediction)[0, 1]),
            "book_equity_slope": float(valuation.coef_[0]),
            "pipeline_version": PIPELINE_VERSION,
        })
        prediction_frame = panel.iloc[test_positions][["permco"]].copy()
        prediction_frame["model"] = model
        prediction_frame["actual"] = y_test
        prediction_frame["predicted"] = prediction
        predictions.append(prediction_frame)
    results = pd.DataFrame(rows)
    panel["sample"] = "train"
    panel.loc[test_positions, "sample"] = "test"
    save_parquet(results, result_path)
    save_parquet(pd.concat(predictions, ignore_index=True), directory / "test_predictions.parquet")
    save_parquet(panel, directory / "evaluation_sample_and_split.parquet")
    return results


quarterly_results = []
for year, quarter in VALIDATION_QUARTERS:
    print(f"Running valuation benchmark for {year}Q{quarter}...")
    quarterly_results.append(run_valuation_benchmark(year, quarter))
quarterly_results = pd.concat(quarterly_results, ignore_index=True)
save_parquet(quarterly_results, VALIDATION_ROOT / "benchmarks" / "three_quarter_results.parquet")
display(quarterly_results.pivot(index="model", columns=["year", "quarter"], values="test_variance_r2"))


## 12. Cross-quarter validation summary and visualization

The paper averages the quarterly variance ratios rather than pooling observations. This final section reports the mean and dispersion across the three validation quarters and saves a diagnostic figure. The three-quarter mean is an integration test, not the final 80-quarter estimate.


In [ ]:
summary = quarterly_results.groupby("model", as_index=False).agg(
    quarters=("quarter", "size"), mean_test_variance_r2=("test_variance_r2", "mean"),
    minimum_test_variance_r2=("test_variance_r2", "min"), maximum_test_variance_r2=("test_variance_r2", "max"),
    standard_deviation_test_r2=("test_variance_r2", "std"), mean_firms=("firms", "mean"),
).sort_values("mean_test_variance_r2", ascending=False).reset_index(drop=True)
save_parquet(summary, VALIDATION_ROOT / "benchmarks" / "three_quarter_summary.parquet")
display(summary)

plot_models = ["placebo_4", "beta", "characteristics_4", "rs_binary_4", "rs_l_min_4", "rs_binary_10", "rs_l_min_10"]
plot_labels = ["Placebo 4D", "Beta", "Four chars.", "RS-Binary 4D", "RS-L-Min 4D", "RS-Binary 10D", "RS-L-Min 10D"]
plot_data = quarterly_results[quarterly_results["model"].isin(plot_models)].copy()
figure, axis = plt.subplots(figsize=(12, 6.5))
x = np.arange(len(plot_models))
width = 0.23
colors = ["#8093A6", "#D19A3E", "#4C956C"]
for offset, ((year, quarter), group) in enumerate(plot_data.groupby(["year", "quarter"], sort=True)):
    values = group.set_index("model").reindex(plot_models)["test_variance_r2"].to_numpy()
    axis.bar(x + (offset - 1) * width, values, width=width, label=f"{year}Q{quarter}", color=colors[offset])
axis.axhline(0, color="#333333", linewidth=0.8)
axis.set_xticks(x, plot_labels, rotation=20, ha="right")
axis.set_ylabel("Out-of-sample relative-valuation $R^2$")
axis.set_title("Three-Quarter Pipeline Validation")
axis.grid(axis="y", alpha=0.25)
axis.legend(frameon=False)
figure.tight_layout()
for extension in ["png", "pdf", "svg"]:
    figure.savefig(VALIDATION_ROOT / "figures" / f"three_quarter_relative_valuation.{extension}", dpi=300, bbox_inches="tight")
plt.show()

print("Three-quarter validation pipeline completed.")
print(f"All validation outputs are stored under: {VALIDATION_ROOT}")


## Validation completion criteria

The pipeline is ready to expand to 80 quarters when:

- all three quarters finish without manual data edits;
- every raw and derived checkpoint can be reloaded after a kernel restart;
- historical CUSIP match rates and sample sizes are economically plausible;
- minimum investor and firm degrees equal 20 after convergence;
- Gaussian placebo performance is approximately zero;
- Beta has low relative-valuation explanatory power;
- the four characteristics have positive explanatory power; and
- holdings embeddings materially outperform the characteristics in most validation quarters.

After these conditions are verified, the production notebook can reuse the same functions with the complete list of quarters from 2005Q1 through 2024Q4.
